# Ionic conductivity of aqueous sodium acetate (1/3): NpT equilibration

⚠️ The three conductivity notebooks are under development and still being tested. ⚠️

This is the first of three notebooks that compute the ionic electrical conductivity
of an aqueous sodium acetate solution:

1. `01_npt_equilibration.ipynb` (this notebook):
   build the initial structure and equilibrate it in the NpT ensemble for 1 ns.
2. `02_nvt_production.ipynb`:
   a 10 ns production run in the NVT ensemble.
   This is the most expensive part and is best run as a job on a GPU.
3. `03_conductivity_analysis.ipynb`:
   visualization and post-processing of the production run with
   [STACIE](https://molmod.github.io/stacie/) to estimate the conductivity and its uncertainty.

These notebooks are designed to produce a reasonable (not very accurate) conductivity estimate at a minimal computational cost.
It is primarily intended as a demonstration of how to use OpenMM and STACIE for ionic conductivity calculations.
For publication quality results, longer simulations with stricter intermediate checks are required.

**Why [sodium acetate](https://en.wikipedia.org/wiki/Sodium_acetate)?**
The acetate anion (CH₃COO⁻) is the simplest carboxylate,
the most common negatively charged group in biomolecules:
it is found in the side chains of aspartate and glutamate,
at the C-terminus of proteins and in many metabolites (lactate, pyruvate, ...).
Unlike sodium chloride, the anion is a small molecule,
which tends to form ion pairs with Na⁺ in molecular dynamics simulations.

**Force field.**
The AMBER force-field files included in OpenMM only contain monatomic ions.
We therefore use CHARMM36 (`charmm36.xml`), which also contains parameters
for many small molecular ions, including acetate.
It also includes specific corrections (NBFIX) for the Lennard-Jones interaction
between Na⁺ and carboxylate oxygens, which reduce the overestimated ion pairing of standard models.

**Conditions.**
We simulate a solution of 10 mass% sodium acetate (1.354 mol/kg) at 20 °C and 1 bar,
for which the following experimental data are available:

- Conductivity: 53.4 mS/cm, from the table "Electrical Conductivity of Aqueous Solutions"
  in the CRC Handbook of Chemistry and Physics
  ([online version](https://hbcp.chemnetbase.com/faces/documents/05_17/05_17_0001.xhtml), [PDF copy](https://diverdi.colostate.edu/all_courses/CRC%20reference%20data/electrical%20conductivity%20of%20aqueous%20solutions.pdf)).
  The data in this table were originally taken from
  A. V. Wolf, *Aqueous Solutions and Body Fluids*, Harper and Row, 1966.
- Density: 1.0495 g/mL, from the
  [Engineering ToolBox](https://engineeringtoolbox.com/density-aqueous-solution-inorganic-sodium-salt-concentration-d_1957.html).

The density is used as a sanity check at the end of this notebook,
while the conductivity is compared to the simulation result in `03_conductivity_analysis.ipynb`.

**Running this notebook.**
You can run this notebook interactively, or as a job with the script `job_npt_vsc_gpu.sh`,
which writes the executed notebook to `01_npt_equilibration.html`.
(Interactive 3D visualizations are not shown in the HTML file.)
For a quick test, reduce `max_step_emin` and `nstep_npt` in the parameter cell below.

In [ ]:
import subprocess
from sys import stdout

import matplotlib.pyplot as plt
import mdtraj
import nglview
import openmm as mm
import pandas
from openmm import app, unit

## 1. Parameters

The number of water molecules is derived from the number of ion pairs and the mass fraction of the salt.
With 40 ion pairs, the simulation cell is rather small (about 5000 atoms),
which keeps the simulations affordable.

In [ ]:
# Composition
num_ion_pairs = 40
mass_fraction = 0.10  # mass of salt / total mass
molar_mass_salt = 82.034  # g/mol, NaCH3COO
molar_mass_water = 18.015  # g/mol
num_water = round(
    num_ion_pairs * molar_mass_salt * (1 - mass_fraction) / mass_fraction / molar_mass_water
)
molality = num_ion_pairs / (num_water * molar_mass_water / 1000)
print(f"Number of water molecules: {num_water}")
print(f"Molality: {molality:.3f} mol/kg")

# Thermodynamic conditions
temperature = 293.15 * unit.kelvin
pressure = 1 * unit.bar

# Rough guess of the density, only used to build the initial structure.
density_guess = 1.05  # g/cm³
# Experimental density, for comparison with the NpT simulation (see introduction).
density_exp = 1.0495  # g/mL

# Maximum number of iterations of the energy minimization.
# (Use 0 for no limit.)
# Default is 100. Reduce to 10 for a quick test.
max_step_emin = 100

# MD settings
timestep = 2 * unit.femtosecond
# 500_000 steps for 1 ns.
# For a quick test, use e.g. 5_000 steps (10 ps).
nstep_npt = 500_000
# Intervals (in steps) for the outputs.
interval_snapshots = 5000  # 5000 steps = 10 ps
interval_scalars = 500  # 500 steps = 1 ps

## 2. Initial structure with Packmol

[Packmol](https://m3g.github.io/packmol/) places the requested numbers of molecules
at random positions in a box, such that atoms of different molecules are at least
2 Å (`tolerance`) apart.
The `pbc` keyword tells Packmol that the box is periodic,
so this condition is also satisfied across the boundaries of the box.
The template structures of the three types of molecules are provided in
`sodium.pdb`, `acetate.pdb` and `water.pdb`.
`acetate.pdb` contains `CONECT` records, which define the bonds in the molecule.
Packmol copies these records to its output file,
so that OpenMM knows which atoms are bonded.

The size of the cubic box is derived from the total mass and the guessed density.

Packmol is executed directly, instead of using the `openff.packmol` Python package, for illustrative purposes.
Both work equally well, but this shows how to run Packmol directly, which can be useful in more specialized setups.

In [ ]:
avogadro = 6.02214076e23  # 1/mol
total_mass = (num_ion_pairs * molar_mass_salt + num_water * molar_mass_water) / avogadro  # g
box_length = (total_mass / density_guess) ** (1 / 3) * 1e8  # cm -> Å
print(f"Box length: {box_length:.2f} Å")

packmol_input = f"""\
tolerance 2.0
filetype pdb
output init_packmol.pdb
seed 1
pbc {box_length:.3f} {box_length:.3f} {box_length:.3f}

structure sodium.pdb
  number {num_ion_pairs}
end structure

structure acetate.pdb
  number {num_ion_pairs}
end structure

structure water.pdb
  number {num_water}
end structure
"""
with open("init_packmol.inp", "w") as f:
    f.write(packmol_input)
print(packmol_input)

# Run Packmol, with the input file as standard input.
with open("init_packmol.inp") as f:
    result = subprocess.run(["packmol"], stdin=f, capture_output=True, text=True, check=False)
if "Success!" not in result.stdout:
    print(result.stdout)
    raise RuntimeError("Packmol failed.")
print("Packmol finished successfully.")

In [ ]:
# Visualize the initial structure: ions as spheres, water as lines.
view = nglview.show_mdtraj(mdtraj.load("init_packmol.pdb"))
view.clear_representations()
view.add_line("water")
view.add_ball_and_stick("ACT")
view.add_spacefill("NA", radius=0.6)
view.add_unitcell()
view

## 3. Force field

OpenMM matches each residue in the topology with a residue template of the force field.
This matching is based on the elements and the bonds of the atoms, not on the names of the residues.
Hence, the residue names `NA`, `ACT` and `HOH` in our PDB file
are correctly matched with the CHARMM36 templates `SOD`, `ACET` and `TIP3`.

Note that we use `charmm36.xml`, not the more recent `charmm36_2024.xml`,
because only the former contains the acetate ion.
The water model is the CHARMM variant of TIP3P (`charmm36/water.xml`),
which also contains the parameters of Na⁺.

The cutoff settings are the standard ones for CHARMM36:
Lennard-Jones interactions are switched off between 1.0 and 1.2 nm,
and long-range electrostatics are treated with PME.
Bonds with hydrogen atoms are constrained, which allows for a time step of 2 fs.

The `System` is saved to an XML file, so that the production run in the next notebook
uses exactly the same force field.
This is done before adding the barostat, which is not used in the production run.

In [ ]:
pdb = app.PDBFile("init_packmol.pdb")
print(pdb.topology)
forcefield = app.ForceField("charmm36.xml", "charmm36/water.xml")
system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.2 * unit.nanometer,
    switchDistance=1.0 * unit.nanometer,
    constraints=app.HBonds,
)
with open("init_system.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(system))

As a sanity check, the following cell prints the net charge of the first residue of each type,
which should be +1 for Na⁺, -1 for acetate and 0 for water.

In [ ]:
nonbonded = next(force for force in system.getForces() if isinstance(force, mm.NonbondedForce))
done = set()
for residue in pdb.topology.residues():
    if residue.name not in done:
        charge = sum(
            nonbonded.getParticleParameters(atom.index)[0].value_in_unit(unit.elementary_charge)
            for atom in residue.atoms()
        )
        print(f"{residue.name:3s} {charge:+.3f} e")
        done.add(residue.name)

## 4. NpT equilibration

The energy is first minimized to remove unfavorable contacts in the Packmol structure.
The number of iterations is limited by `max_step_emin`,
because a full minimization is not needed: the remaining strain is quickly removed by the thermostat.
Then the system is equilibrated with a Langevin thermostat and a Monte Carlo barostat.
The Langevin thermostat is very effective for equilibration,
but it perturbs the dynamics of the atoms, which is why it is not used in the production run.

In [ ]:
# System configuration and energy minimization
system.addForce(mm.MonteCarloBarostat(pressure, temperature))
integrator = mm.LangevinMiddleIntegrator(temperature, 1 / unit.picosecond, timestep)
simulation = app.Simulation(pdb.topology, system, integrator)
print("Platform:", simulation.context.getPlatform().getName())
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy(maxIterations=max_step_emin)
simulation.context.setVelocitiesToTemperature(temperature)

In [ ]:
# Snapshots for visualization.
simulation.reporters.append(app.DCDReporter("traj_npt.dcd", interval_snapshots))
# Scalar properties for plotting, and the performance of the simulation.
simulation.reporters.append(
    app.StateDataReporter(
        "scalars_npt.csv",
        interval_scalars,
        time=True,
        potentialEnergy=True,
        temperature=True,
        volume=True,
        density=True,
        speed=True,
        remainingTime=True,
        totalSteps=nstep_npt,
    )
)
# Progress on screen, at most 20 lines.
simulation.reporters.append(
    app.StateDataReporter(
        stdout,
        max(nstep_npt // 20, 1),
        step=True,
        temperature=True,
        density=True,
        progress=True,
        remainingTime=True,
        speed=True,
        totalSteps=nstep_npt,
    )
)
simulation.step(nstep_npt)

## 5. Saving the final state

The final positions, velocities and box vectors are written to an XML file,
which is used to start the production run in the next notebook.
Unlike a checkpoint file, an XML file can be loaded on any type of hardware,
so you can, for example, run this notebook on a CPU and the production run on a GPU.

The final structure is also written to a PDB file, which defines the topology of the production run.
For this file, all molecules are put back into the simulation cell (`enforcePeriodicBox=True`),
which gives a nicer visualization.

In [ ]:
state = simulation.context.getState(getPositions=True, getVelocities=True)
with open("init_nvt_state.xml", "w") as f:
    f.write(mm.XmlSerializer.serialize(state))

state_wrapped = simulation.context.getState(getPositions=True, enforcePeriodicBox=True)
simulation.topology.setPeriodicBoxVectors(state.getPeriodicBoxVectors())
with open("init_nvt.pdb", "w") as f:
    app.PDBFile.writeFile(simulation.topology, state_wrapped.getPositions(), f)

## 6. Inspection of the equilibration run

The temperature and the density should fluctuate around stable values after a short transient.
The average density over the second half of the run is printed below
and compared with the experimental value.
A deviation of 1 to 2% is not unusual for this type of force field,
but much larger deviations would point to a problem with the force field or the simulation settings.

In [ ]:
df = pandas.read_csv("scalars_npt.csv")
df = df.rename(columns={'#"Time (ps)"': "Time (ps)"})
plt.close("dens")
fig, axs = plt.subplots(2, 1, sharex=True, figsize=(6, 5), num="dens")
df.plot(x="Time (ps)", y="Temperature (K)", ax=axs[0], legend=False)
axs[0].set_ylabel("Temperature (K)")
df.plot(x="Time (ps)", y="Density (g/mL)", ax=axs[1], legend=False)
axs[1].set_ylabel("Density (g/mL)")
fig.tight_layout()

second_half = df["Density (g/mL)"][len(df) // 2 :]
print(f"Average density (second half): {second_half.mean():.4f} g/mL")
print(f"Experimental density: {density_exp:.4f} g/mL")
print(f"Relative deviation: {(second_half.mean() / density_exp - 1) * 100:+.1f}%")
print(f"Final box length: {state.getPeriodicBoxVectors()[0][0]}")

In [ ]:
traj = mdtraj.load("traj_npt.dcd", top="init_nvt.pdb")
view = nglview.show_mdtraj(traj)
view.clear_representations()
view.add_line("water")
view.add_ball_and_stick("ACT")
view.add_spacefill("NA", radius=0.6)
view.add_unitcell()
view